# LAB 2: Semantic Network

## Objectives

1. To represent semantic networks and knowledge representation.
2. To understand POS tagging and identify grammatical categories of words.
3. To learn Named Entity Recognition (NER) for extracting entities such as persons, organizations, and locations from text.
4. To understand vector embeddings.
5. To learn the concept of calculating Jaccard Similarity.
6. To understand how similarity measures are used in recommendation systems.


# Theory

## Semantic Network:

A semantic network is a knowledge representation structure that depicts how concepts are related to one another and how they interconnect. It maps relationships between different concepts to represent knowledge in Artificial Intelligence (AI) systems.

Semantic networks use nodes to represent concepts and links to represent relationships between those concepts. They help AI systems organize and retrieve knowledge efficiently.


## Part of Speech (POS) Tagging

POS tagging is the process of assigning a grammatical category (part of speech) to each word in a sentence. It is a type of classification where each word is labeled according to its role in the sentence.

### Types of Parts of Speech

### 1. Noun

A noun names a person, place, thing, or idea.

**Example:**

Ram is a student.

### 2. Pronoun

A pronoun is used in place of a noun.

**Examples:**

Ram and Hari are playing football.

They are playing football.

### 3. Verb

A verb shows an action or state of being.

**Examples:**

She writes a letter.

They are happy.

### 4. Adjective

An adjective describes a noun or pronoun.

**Examples:**

It is a beautiful flower.

He is a smart student.

### 5. Adverb

An adverb modifies a verb, adjective, or another adverb.

**Examples:**

She sings beautifully.

He runs very fast.

### 6. Preposition

A preposition shows the relationship between a noun or pronoun and another word.

**Examples:**

The book is on the table.

She went to school.

### 7. Conjunction

A conjunction joins words, phrases, or clauses.

**Examples:**

Ram and Shyam are friends.

I tried hard, but I failed.

### 8. Interjection

An interjection expresses sudden emotion.

**Example:**

Wow! What a beautiful view.


## Named Entity Recognition (NER):

Named Entity Recognition (NER) is a Natural Language Processing technique used to identify and classify important entities in text, such as persons, organizations, and locations.

NER helps analyze unstructured text and reduces the need for manual analysis when dealing with large amounts of data.

### Example

Sentence:

Ram is the General Manager of XYZ Bank in Kathmandu.

**Entities:**

- Ram → Person
- XYZ Bank → Organization
- Kathmandu → Location



## Vector Embeddings:

Vector Embedding is a technique in Natural Language Processing (NLP) and Machine Learning that converts words, sentences, images, or other data into numerical vectors while preserving their meaning and relationships.

### Example

dog → [0.10, 0.20, ...]

cat → [0.11, 0.22, ...]

Words with similar meanings have vectors that are located close to each other in vector space.



## Knowledge Triples:

A Knowledge Triple is a structured representation of a fact in the form:

**(Subject, Predicate, Object)**

where the subject and object are entities and the predicate describes the relationship between them.

### Example

Sentence:

Ram has a dog.

**Subject:** Ram

**Predicate:** has

**Object:** dog

Knowledge triples are widely used in knowledge graphs and semantic networks to represent information in a structured manner.

In [ ]:
"""
Academic Paper Recommendation System using Jaccard Similarity
=============================================================
Recommends papers based on shared features:
  - Domain (CV, NLP, Geometric Deep Learning)
  - Methodologies utilized
  - Authors
  - Citation links (cites / cited-by)

Usage:
    python jaccard_recommender.py
    python jaccard_recommender.py --file path/to/network_data.json
"""

import json
import argparse
import os

# ─────────────────────────────────────────────
# STEP 1: Load JSON file
# ─────────────────────────────────────────────

def load_data(filepath):
    """Load nodes and links from a JSON file."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Could not find file: {filepath}")
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    nodes = data.get("nodes", [])
    links = data.get("links", [])
    print(f"  Loaded {len(nodes)} nodes and {len(links)} links from '{filepath}'")
    return nodes, links


# ─────────────────────────────────────────────
# STEP 2: Filter node groups
# ─────────────────────────────────────────────

def get_group(nodes, group_name):
    return [n["id"] for n in nodes if n.get("group") == group_name]


# ─────────────────────────────────────────────
# STEP 3: Graph lookup helpers
# ─────────────────────────────────────────────

def get_domain(paper_id, links):
    for l in links:
        if l["source"] == paper_id and l["label"] == "BELONGS_TO":
            return l["target"]
    return None

def get_methods(paper_id, links):
    return [l["target"] for l in links if l["source"] == paper_id and l["label"] == "UTILIZED"]

def get_authors(paper_id, links):
    return [l["source"] for l in links if l["target"] == paper_id and l["label"] == "AUTHORED"]

def get_citations(paper_id, links):
    return [l["target"] for l in links if l["source"] == paper_id and l["label"] == "CITES"]

def get_cited_by(paper_id, links):
    return [l["source"] for l in links if l["target"] == paper_id and l["label"] == "CITES"]


# ─────────────────────────────────────────────
# STEP 4: Build feature set for a paper
# ─────────────────────────────────────────────

def build_features(paper_id, links):
    """
    Represent a paper as a set of tagged feature strings:
      'domain:Computer Vision'
      'method:Transformer Architectures'
      'author:Dr_A_Vaswani'
      'cites:BERT-Large: ...'
      'citedby:GPT-Next: ...'
    """
    features = set()

    domain = get_domain(paper_id, links)
    if domain:
        features.add(f"domain:{domain}")

    for m in get_methods(paper_id, links):
        features.add(f"method:{m}")

    for a in get_authors(paper_id, links):
        features.add(f"author:{a}")

    for c in get_citations(paper_id, links):
        features.add(f"cites:{c}")

    for cb in get_cited_by(paper_id, links):
        features.add(f"citedby:{cb}")

    return features


# ─────────────────────────────────────────────
# STEP 5: Jaccard Similarity
# ─────────────────────────────────────────────

def jaccard_similarity(set_a, set_b):
    """
    Jaccard Similarity = |A ∩ B| / |A ∪ B|

    Returns a float between 0.0 (no overlap) and 1.0 (identical).
    """
    intersection = set_a & set_b
    union = set_a | set_b
    if not union:
        return 0.0
    return len(intersection) / len(union)


# ─────────────────────────────────────────────
# STEP 6: Recommendation functions
# ─────────────────────────────────────────────

def recommend_by_paper(paper_id, papers, links, top_n=5):
    """Recommend papers similar to a given paper."""
    if paper_id not in papers:
        print(f"  [!] Paper not found: {paper_id}")
        return []
    target_features = build_features(paper_id, links)
    scores = []
    for p in papers:
        if p == paper_id:
            continue
        score = jaccard_similarity(target_features, build_features(p, links))
        if score > 0:
            scores.append((p, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_n]


def recommend_by_author(author_id, authors, papers, links, top_n=5):
    """Recommend papers for an author based on their research profile."""
    if author_id not in authors:
        print(f"  [!] Author not found: {author_id}")
        return []
    authored = set(
        l["target"] for l in links
        if l["source"] == author_id and l["label"] == "AUTHORED"
    )
    author_features = set()
    for p in authored:
        author_features |= build_features(p, links)
    author_features.add(f"author:{author_id}")

    scores = []
    for p in papers:
        if p in authored:
            continue
        score = jaccard_similarity(author_features, build_features(p, links))
        if score > 0:
            scores.append((p, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_n]


def recommend_by_method(method_id, methods, papers, links, top_n=5):
    """Recommend papers that use a given methodology."""
    if method_id not in methods:
        print(f"  [!] Method not found: {method_id}")
        return []
    method_features = {f"method:{method_id}"}
    scores = []
    for p in papers:
        score = jaccard_similarity(method_features, build_features(p, links))
        if score > 0:
            scores.append((p, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_n]


# ─────────────────────────────────────────────
# STEP 7: Pretty-print helper
# ─────────────────────────────────────────────

def print_recommendations(title, results, links):
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")
    if not results:
        print("  No recommendations found.")
        return
    for rank, (paper, score) in enumerate(results, start=1):
        bar_len = int(score * 30)
        bar = "█" * bar_len + "░" * (30 - bar_len)
        domain = get_domain(paper, links) or "Unknown"
        short_domain = (domain
            .replace("Computer Vision", "CV")
            .replace("Natural Language Processing", "NLP")
            .replace("Geometric Deep Learning", "GDL"))
        print(f"  {rank}. [{bar}] {score:.4f}  [{short_domain}]")
        print(f"     {paper}")


# ─────────────────────────────────────────────
# STEP 8: Interactive mode
# ─────────────────────────────────────────────

def interactive_mode(papers, authors, methods, links):
    print(f"\n{'='*70}")
    print("  INTERACTIVE MODE  (type 'quit' to exit)")
    print(f"{'='*70}")
    print("\n  Modes:  [1] By Paper   [2] By Author   [3] By Method\n")

    while True:
        mode = input("  Choose mode (1/2/3) or 'quit': ").strip()
        if mode.lower() == 'quit':
            print("\n  Goodbye!\n")
            break

        elif mode == '1':
            print("\n  Available papers:")
            for i, p in enumerate(sorted(papers), 1):
                print(f"    {i:2}. {p}")
            query = input("\n  Enter paper title (or number): ").strip()
            if query.isdigit():
                idx = int(query) - 1
                query = sorted(papers)[idx] if 0 <= idx < len(papers) else query
            results = recommend_by_paper(query, papers, links, top_n=5)
            print_recommendations(f"Similar to: {query}", results, links)

        elif mode == '2':
            print("\n  Available authors:", ", ".join(
                a.replace("Dr_", "Dr. ").replace("_", " ") for a in authors))
            query = input("  Enter author ID (e.g. Dr_Y_Bengio): ").strip()
            results = recommend_by_author(query, authors, papers, links, top_n=5)
            print_recommendations(f"Recommended for: {query}", results, links)

        elif mode == '3':
            print("\n  Available methods:", ", ".join(methods))
            query = input("  Enter method name: ").strip()
            results = recommend_by_method(query, methods, papers, links, top_n=5)
            print_recommendations(f"Papers using: {query}", results, links)

        else:
            print("  Invalid choice. Enter 1, 2, 3, or 'quit'.")


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

if __name__ == "__main__":

    # --- Argument parser: lets you pass a custom file path ---
    parser = argparse.ArgumentParser(description="Paper Recommendation via Jaccard Similarity")
    parser.add_argument(
        "--file", "-f",
        type=str,
        default="network_data.json",          # default: same folder as script
        help="Path to your JSON file (default: network_data.json)"
    )
    args = parser.parse_args()

    # --- Load data ---
    nodes, links = load_data(args.file)

    # --- Separate node types ---
    papers  = get_group(nodes, "Paper")
    authors = get_group(nodes, "Author")
    methods = get_group(nodes, "Methodology")

    print(f"\n★{'★'*68}★")
    print("  ACADEMIC PAPER RECOMMENDATION SYSTEM — Jaccard Similarity")
    print(f"★{'★'*68}★")

    # --- Demo runs ---
    if papers:
        results = recommend_by_paper(papers[0], papers, links, top_n=5)
        print_recommendations(f"Papers similar to:\n  » {papers[0]}", results, links)

    if authors:
        results = recommend_by_author(authors[0], authors, papers, links, top_n=5)
        print_recommendations(f"Recommended for author: {authors[0]}", results, links)

    if methods:
        results = recommend_by_method(methods[0], methods, papers, links, top_n=5)
        print_recommendations(f"Papers using methodology: {methods[0]}", results, links)

    # --- Launch interactive mode ---
    interactive_mode(papers, authors, methods, links)

  Loaded 0 nodes and 0 links from '/root/.local/share/jupyter/runtime/kernel-2e9523fd-3dc9-4c3d-ae0b-8b72d263a101.json'

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  ACADEMIC PAPER RECOMMENDATION SYSTEM — Jaccard Similarity
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

  INTERACTIVE MODE  (type 'quit' to exit)

  Modes:  [1] By Paper   [2] By Author   [3] By Method

  Choose mode (1/2/3) or 'quit': 1

  Available papers:

  Enter paper title (or number): 3
  [!] Paper not found: 3

  Similar to: 3
  No recommendations found.
  Choose mode (1/2/3) or 'quit': quit

  Goodbye!



## Discussion

In this lab, we studied the concept of semantic networks and knowledge representation. We learned how information can be represented using nodes and relationships. We also explored Part of Speech (POS) tagging and Named Entity Recognition (NER), which help in understanding the grammatical structure of text and identifying important entities such as people, organizations, and locations.

We also learned about vector embeddings, which convert text into numerical vectors while preserving semantic meaning. The concept of knowledge triples was used to represent facts in a structured format consisting of subject, predicate, and object. Finally, Jaccard Similarity was applied to compare the similarity between papers based on their features and to generate recommendations. This demonstrated how similarity measures are used in recommendation systems.

## Conclusion

This lab helped us understand the fundamentals of semantic networks and knowledge representation. We learned the applications of POS tagging, Named Entity Recognition, vector embeddings, and knowledge triples in Natural Language Processing. We also understood how Jaccard Similarity can be used to measure similarity between entities and support recommendation systems. Overall, the lab provided practical insight into how AI systems process, organize, and recommend information efficiently.